# Module 1 — Mô hình Ridge Regression (Cửa hàng × Nhóm hàng × Ngày)

Dự báo `so_luong` mỗi ngày cho từng cặp cửa hàng × nhóm hàng bằng **hồi quy Ridge** (bình phương tối thiểu có phạt L2). Hệ số co về 0 khi các đặc trưng lag/rolling tương quan mạnh.

- **Đầu vào:** `outputs/daily_store_category.csv` (từ `module3/Module3_1_tien_xu_ly.ipynb`) + đặc trưng `tao_dac_trung` của Module 3.
- **So sánh công bằng:** cùng bảng, cùng `FEATS`, cùng mốc chia (`chia_tap`), cùng chỉ số (`danh_gia`) và baseline (`bang_baseline`) với các mô hình khác.
- **Xử lý riêng của mô hình tuyến tính** (cùng các cột, không thêm thông tin mới): `thu` và ba cột mã được mã hoá one-hot thay vì coi là số thứ tự; các đặc trưng lượng bán gần đây có thể lấy `log1p` (chọn trên Validation); mọi cột số được chuẩn hoá.
- **Đầu ra** `outputs/ridge/`: `ridge_tuning.csv`, `ridge_best_params.json`, `ridge_metrics.csv`, `ridge_model.joblib`, `ridge_predictions.csv`, `ridge_feature_importance.*`, `ridge_error_by_category.csv`, `ridge_error_by_store.csv`, `ridge_thuc_te_vs_du_bao.png`.

**Cấu hình** (`.env` ở gốc repo, mẫu `.env.example`). **Chạy trên Google Colab:** upload `module3_preprocess.py` và `.env` cùng thư mục với notebook, rồi Runtime → Run all.

In [ ]:
import os, sys, json, time, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline

sys.path.append("../module3")
from module3_preprocess import (OUT_DIR as OUT, SEED, doc_bang_ngay, tao_dac_trung, chia_tap, danh_gia,
                                bang_baseline, loi_theo_nhom, nop_ket_qua, tai_ve)

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
OUT_R = f"{OUT}/ridge"; os.makedirs(OUT_R, exist_ok=True)
daily = doc_bang_ngay()
print(f"Bảng mô hình: {len(daily):,} dòng")

## 1. Đặc trưng và chia tập

Dùng nguyên `FEATS` của Module 3, chia thành 3 nhóm để tiền xử lý cho mô hình tuyến tính:

- **Lượng bán gần đây** (`lag_1/7/14`, `tb_7/14`, `std_7/14`): chuẩn hoá; có thể lấy `log1p` trước (chọn ở mục 2).
- **Lịch dạng số** (`cuoi_tuan`, `ngay_trong_thang`, `ngay_toi_tet`): chuẩn hoá.
- **Phân loại** (`thu`, `Mã cửa hàng_code`, `Nhóm hàng_code`, `Vùng_code`): one-hot — mã cửa hàng không có thứ tự nên không dùng như một số.

In [ ]:
d, FEATS = tao_dac_trung(daily)
SO = ["lag_1", "lag_7", "lag_14", "tb_7", "tb_14", "std_7", "std_14"]
LICH = ["cuoi_tuan", "ngay_trong_thang", "ngay_toi_tet"]
PL = ["thu", "Mã cửa hàng_code", "Nhóm hàng_code", "Vùng_code"]
assert sorted(SO + LICH + PL) == sorted(FEATS)
fit, val, tr, te = chia_tap(d)

base_val = pd.DataFrame(bang_baseline(val)).T
rows = bang_baseline(te)
print("Baseline trên Validation:"); display(base_val)
print("Baseline trên Test:"); display(pd.DataFrame(rows).T)

## 2. Tối ưu tham số trên Validation

- Lưới 12 cấu hình: `alpha` (mức phạt L2) ∈ {10⁻³, 10⁻², 10⁻¹, 1, 10, 100} × biến đổi lượng bán ∈ {`log1p`, `goc`}.
- Train trên 15/01–14/02, chấm trên Validation 15–28/02; **chọn cấu hình theo MAE trên Validation** — cùng tiêu chí với các mô hình khác.

In [ ]:
def dung_mo_hinh(alpha, bien_doi):
    """Pipeline: tiền xử lý theo nhóm cột → Ridge (phạt L2)."""
    so = (make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler())
          if bien_doi == "log1p" else StandardScaler())
    pre = ColumnTransformer([("so", so, SO), ("lich", StandardScaler(), LICH),
                             ("pl", OneHotEncoder(handle_unknown="ignore", sparse_output=False), PL)])
    return Pipeline([("pre", pre), ("m", Ridge(alpha=alpha, random_state=SEED))])

params = [{"alpha": a, "bien_doi": b} for b in ["log1p", "goc"] for a in [1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0]]
log = []
for i, p in enumerate(params):
    t = time.time()
    m = dung_mo_hinh(**p).fit(fit[FEATS], fit["so_luong"])
    s = danh_gia(val["so_luong"], np.clip(m.predict(val[FEATS]), 0, None))
    log.append({"id": i, **p, **s, "giây": round(time.time() - t, 1)})
    print(f"{i + 1:2}/{len(params)} MAE={s['MAE']:.4f} WAPE={s['WAPE']:.3f} Bias={s['Bias']:+.1%} {p}")
tuning = pd.DataFrame(log).sort_values("MAE").reset_index(drop=True)
tuning.to_csv(f"{OUT_R}/ridge_tuning.csv", index=False, encoding="utf-8-sig")
best = params[int(tuning.loc[0, "id"])]
with open(f"{OUT_R}/ridge_best_params.json", "w", encoding="utf-8") as f:
    json.dump({**best, "one_hot": PL, "chuan_hoa": SO + LICH}, f, indent=2, ensure_ascii=False)
print("Tốt nhất:", best)
display(tuning.head(5))

## 3. Train lại trên toàn bộ Train và chấm Test

Train lại cấu hình tốt nhất trên 15/01–28/02 rồi chấm trên tháng 3. Dự báo âm được cắt về 0.

In [ ]:
ridge = dung_mo_hinh(**best).fit(tr[FEATS], tr["so_luong"])
te["du_bao_ridge"] = np.clip(ridge.predict(te[FEATS]), 0, None)
rows["Ridge"] = danh_gia(te["so_luong"], te["du_bao_ridge"])
result = pd.DataFrame(rows).T
result["WAPE so với MA7"] = result["WAPE"] / result.loc["MA7", "WAPE"] - 1
result.to_csv(f"{OUT_R}/ridge_metrics.csv", encoding="utf-8-sig")
joblib.dump(ridge, f"{OUT_R}/ridge_model.joblib", compress=3)
display(result)

## 4. Đặc trưng quan trọng và lỗi theo nhóm hàng / cửa hàng / ngày

**Mức quan trọng** theo chuẩn chung cho Ridge/Poisson là trị tuyệt đối hệ số trên đặc trưng đã chuẩn hoá. Với cột one-hot (một đặc trưng gốc thành nhiều cột 0/1): **độ lệch chuẩn phần đóng góp của đặc trưng vào dự báo** trên tập Train — với cột số đã chuẩn hoá, giá trị này đúng bằng |β|.

In [ ]:
pre, mdl = ridge.named_steps["pre"], ridge.named_steps["m"]
X = pre.transform(tr[FEATS]); ten_cot = pre.get_feature_names_out()
nguon = [next(f for f in FEATS if c.split("__", 1)[1] == f or c.split("__", 1)[1].startswith(f + "_")) for c in ten_cot]
dong_gop = pd.DataFrame(X * mdl.coef_, columns=ten_cot).T.groupby(nguon).sum().T
imp = dong_gop.std().reindex(FEATS).sort_values()
imp.to_csv(f"{OUT_R}/ridge_feature_importance.csv", header=["importance"], encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(7, 5)); ax.barh(imp.index, imp.values, color="#2a78d6")
ax.set_title("Ridge — mức độ quan trọng của đặc trưng"); fig.tight_layout()
fig.savefig(f"{OUT_R}/ridge_feature_importance.png", dpi=150); plt.show()

err_cat = loi_theo_nhom(te, "du_bao_ridge", "Nhóm hàng")
err_store = loi_theo_nhom(te, "du_bao_ridge", ["Mã cửa hàng", "Khu vực"])
err_cat.to_csv(f"{OUT_R}/ridge_error_by_category.csv", encoding="utf-8-sig")
err_store.to_csv(f"{OUT_R}/ridge_error_by_store.csv", encoding="utf-8-sig")
display(err_cat)
print(f"Ridge có WAPE thấp hơn MA7 ở {(err_cat.WAPE < err_cat.WAPE_MA7).sum()}/{len(err_cat)} nhóm hàng "
      f"và {(err_store.WAPE < err_store.WAPE_MA7).sum()}/{len(err_store)} cửa hàng")

day = te.groupby("Ngày")[["so_luong", "du_bao_ridge", "tb_7"]].sum()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(day.index, day["so_luong"], color="#1f2a2a", lw=2, label="Thực tế")
ax.plot(day.index, day["du_bao_ridge"], color="#2a78d6", lw=2, label="Ridge")
ax.plot(day.index, day["tb_7"], color="#898781", lw=1.5, ls="--", label="MA7")
ax.set_title("Tổng số lượng tháng 3 — thực tế và dự báo"); ax.legend(); ax.grid(axis="y", alpha=.3)
fig.tight_layout(); fig.savefig(f"{OUT_R}/ridge_thuc_te_vs_du_bao.png", dpi=150); plt.show()
te[["Mã cửa hàng", "Nhóm hàng", "Khu vực", "Vùng", "Ngày", "so_luong", "du_bao_ridge", "lag_1", "lag_7", "tb_7"]] \
    .to_csv(f"{OUT_R}/ridge_predictions.csv", index=False, encoding="utf-8-sig")

## 5. Kết quả và thảo luận

*Số liệu từ lần chạy ngày 12/10/2026 trên bảng mô hình chưa lọc (giữ bán buôn, online, trả lại) — cùng bảng với Poisson, Random Forest và LightGBM. Test tháng 3 gồm 67.146 ô.*

**Tối ưu tham số.** Cấu hình tốt nhất trên Validation: `alpha=100`, đặc trưng lượng bán để nguyên (`goc`) — MAE Validation 0,753, **tốt hơn MA7 (0,847)**. Biến đổi `log1p` làm MAE tăng lên khoảng 1,06–1,09; trên thang gốc, `alpha` từ 0,001 đến 100 chỉ đổi MAE Validation 0,002 (phạt L2 gần như không ảnh hưởng khi các cột số đã chuẩn hoá).

| Cấu hình (3 tốt nhất theo MAE Val) | MAE Val | RMSE Val | WAPE Val | Bias Val |
|---|---:|---:|---:|---:|
| alpha 100 · goc | **0,753** | **8,797** | **0,666** | **−13,6%** |
| alpha 10 · goc | 0,755 | 8,822 | 0,668 | −13,8% |
| alpha 1 · goc | 0,755 | 8,822 | 0,668 | −13,8% |
| *MA7 (tham chiếu)* | *0,847* | *8,474* | *0,749* | *+11,2%* |

**Kết quả trên Test (tháng 3, 67.146 ô):**

| Mô hình | MAE | RMSE | WAPE | R² | Bias |
|---|---:|---:|---:|---:|---:|
| Naive (lag 1) | 0,887 | 6,675 | 0,802 | −0,142 | −0,9% |
| Seasonal Naive (lag 7) | 0,908 | 6,728 | 0,821 | −0,160 | +1,6% |
| MA7 | **0,744** | **4,975** | **0,674** | **0,365** | **−0,2%** |
| **Ridge** | 0,802 | 4,984 | 0,725 | 0,363 | +10,7% |

**Thảo luận.**

1. **Ridge thua MA7 trên mọi chỉ số Test.** MAE 0,802 so với 0,744, WAPE 0,725 so với 0,674 (kém 7,6%), RMSE 4,984 so với 4,975, R² 0,363 so với 0,365. Trên Validation Ridge lại thắng MA7 rõ (MAE 0,753 so với 0,847): Validation 15–28/02 là giai đoạn phục hồi sau Tết, không đại diện cho tháng 3. Bias cũng đổi dấu, từ −13,6% trên Validation sang +10,7% trên Test.
2. **Dư tổng lượng, nhưng MAE ở ô bán lớn vẫn thấp hơn MA7.** Bias +10,7%: trung bình 2.650 sản phẩm/ngày toàn hệ thống so với thực tế 2.393 (MA7 2.388); tổng dự báo 82.157 so với 74.196. Ở 278 ô bán ≥ 20 sản phẩm/ngày (thực tế 13.570), Ridge dự báo 9.810 — thấp hơn MA7 (10.904) — nhưng MAE trên các ô này là 27,807 so với 30,331 của MA7. Phần thua MAE toàn cục nằm ở ô nhỏ: 46.477 ô bán 0 được dự báo tổng 16.295 (MAE 0,351 so với 0,241).
3. **Thắng MA7 ở 2/23 nhóm hàng và 12/130 cửa hàng.** Hai nhóm thắng là Điện thoại (WAPE 0,538 so với 0,559) và Phụ kiện (0,500 so với 0,507), chiếm 59,6% lượng bán tháng 3. Hcare thua (0,562 so với 0,552). Phần thắng ở hai nhóm lớn không bù sai số ở 21 nhóm còn lại.
4. **Đặc trưng quan trọng:** trung bình 7 ngày (33,8%), độ lệch chuẩn 7 ngày (16,0%), trung bình 14 ngày (13,5%), lag 1 (10,9%). Bảy đặc trưng lượng bán gần đây cộng 90,0%; ba cột mã cửa hàng/nhóm/vùng cộng 5,6%; lịch 4,4%. Mô hình tuyến tính chủ yếu nội suy mức bán gần đây, ít dùng mã cửa hàng hay nhóm hàng — khác Poisson (ba cột mã chiếm phần lớn mức quan trọng).
5. **So với các mô hình đã có** (cùng bảng, đặc trưng, mốc chia): Ridge có MAE 0,802, sát Poisson (0,799) và cao hơn Random Forest (0,757) lẫn LightGBM (0,678). |Bias| = 10,7% vượt ngưỡng 10% của tiêu chí đề xuất, trong khi Poisson (+5,7%) và Random Forest (+0,8%) vẫn nằm trong ngưỡng nhưng cả hai chưa thắng MA7 về MAE. LightGBM thắng MAE/WAPE nhưng thiếu 37,3% tổng lượng.

**Giới hạn:** bảng mô hình chưa lọc bán buôn/online; mô hình tuyến tính không học tương tác giữa nhóm hàng và lịch; một mốc chia; Validation không đại diện cho Test. `alpha` trên thang gốc gần như không đổi MAE, nên mở rộng lưới phạt L2 không phải hướng chính.

## Nộp kết quả vào báo cáo và tải về (Colab)

`nop_ket_qua` lọc các kết quả đưa vào báo cáo sang `bao_cao/du_lieu/ridge/`. Sau đó chạy `python module5/tao_bao_cao.py` để cập nhật `bao_cao/BAO_CAO.md` rồi commit `bao_cao/` (trên Colab: giải nén `bao_cao_ridge.zip` vào `bao_cao/du_lieu/ridge/` trong repo trước).

In [ ]:
nop_ket_qua("ridge", val)
tai_ve(OUT_R, "module1_ridge_outputs")